In [1]:
from finn.util.visualization import showInNetron
import onnx
from qonnx.util.basic import qonnx_make_model
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp
import os

In [2]:
root_dir = f'{os.getcwd()}/../../local_files'

chr_rtlsim_model_path = f'/{root_dir}/VGG10_build_output/run_2_characterize_rtlsim/intermediate_models/step_set_fifo_depths.onnx'
chr_analytical_model_path = f'/{root_dir}/VGG10_build_output/run_3_characterize_analytical/intermediate_models/step_generate_estimate_reports.onnx'
rtlsim_model_path = f'/{root_dir}/VGG10_build_output/run_0_largefifo_rtlsim/intermediate_models/step_set_fifo_depths.onnx'

In [ ]:
showInNetron(chr_analytical_model_path)

In [ ]:
root_dir = f'{os.getcwd()}/../../local_files'

chr_rtlsim_model_path = f'/{root_dir}/VGG10_build_output/run_2_characterize_rtlsim/intermediate_models/step_set_fifo_depths.onnx'
chr_analytical_model_path = f'/{root_dir}/VGG10_build_output/run_3_characterize_analytical/intermediate_models/step_set_fifo_depths.onnx'
rtlsim_model_path = f'/{root_dir}/VGG10_build_output/run_0_largefifo_rtlsim/intermediate_models/step_set_fifo_depths.onnx'

chr_rtlsim_model = ModelWrapper(chr_rtlsim_model_path)
chr_analytical_model = ModelWrapper(chr_analytical_model_path)
rtlsim_model = ModelWrapper(rtlsim_model_path)

def get_depth(model,i):
    inst = getCustomOp(model.graph.node[i])
    try:
        depth = inst.get_nodeattr("depth")
    except:
        depth = None
    return model.graph.node[i].name,depth

print(f'{str("Input Node"):<40} to {str("Output Node"):<40} | {str("largefifos"):<10} {str("rtlsim_chr"):<10} {str("analytical"):<10}')
for i in range(len(chr_model.graph.node)):
    chr_rtlsim_name,chr_rtlsim_depth = get_depth(chr_rtlsim_model,i)
    chr_analytical_name,chr_analytical_depth = get_depth(chr_analytical_model,i)
    rtlsim_name,rtlsim_depth = get_depth(rtlsim_model,i)
    if chr_rtlsim_depth != None:
        if i > 0:
            input_node = chr_rtlsim_model.graph.node[i-1].name
        else:
            input_node = "global_in"
        if i < len(chr_model.graph.node)-1:
            output_node = chr_rtlsim_model.graph.node[i+1].name
        else:
            output_node = "global_out"
        print(f'{str(input_node or "N/A"):<40} to {str(output_node or "N/A"):<40} | {str(rtlsim_depth or "N/A"):<10} {str(chr_rtlsim_depth or "N/A"):<10} {str(chr_analytical_depth or "N/A"):<10}')


In [ ]:
getCustomOp(chr_analytical_model.graph.node[3]).get_nodeattr("outFIFODepths")

In [3]:
# fifo size analysis per-node

root_dir = f'{os.getcwd()}/../../local_files'

chr_rtlsim_model_path = f'/{root_dir}/VGG10_build_output/run_2_characterize_rtlsim/intermediate_models/step_set_fifo_depths.onnx'
chr_analytical_model_path = f'/{root_dir}/VGG10_build_output/run_3_characterize_analytical/intermediate_models/step_set_fifo_depths.onnx'
rtlsim_model_path = f'/{root_dir}/VGG10_build_output/run_0_largefifo_rtlsim/intermediate_models/step_set_fifo_depths.onnx'

chr_rtlsim_model_original = ModelWrapper(chr_rtlsim_model_path)
chr_analytical_model_original = ModelWrapper(chr_analytical_model_path)
rtlsim_model_original = ModelWrapper(rtlsim_model_path)


def get_fifo_table(rtlsim_model, chr_rtlsim_model, chr_analytical_model):
    def get_depths(model,i):
        inst = getCustomOp(model.graph.node[i])
        if "FIFO" not in model.graph.node[i].name:
            in_depth = inst.get_nodeattr("inFIFODepths")
            out_depth = inst.get_nodeattr("outFIFODepths")
            return (model.graph.node[i].name,in_depth[0], out_depth[0])
        else:
            return None
    
    def get_all_depths(model):
        l = []
        for i in range(len(model.graph.node)):
            depth = get_depths(model,i)
            if depth is not None:
                l.append(depth)
        return l

    chr_rtlsim_list = get_all_depths(chr_rtlsim_model)
    chr_analytical_list = get_all_depths(chr_analytical_model)
    rtlsim_list = get_all_depths(rtlsim_model)

    string = ""
    string += "inFIFODepths:\n"
    string += f'{str("node"):<40} | {str("rtlsim_large"):<13} | {str("chr_rtlsim"):<13} | {str("chr_analytical"):<13}\n'
    
    for i in range(len(chr_rtlsim_list)):
        string += f'{str(chr_rtlsim_list[i][0] or "N/A"):<40} | {str(rtlsim_list[i][1]):<13} | {str(chr_rtlsim_list[i][1]):<13} | {str(chr_analytical_list[i][1]):<13}\n'
    
    
    string += "outFIFODepths:\n"
    string += f'{str("node"):<40} | {str("rtlsim_large"):<13} | {str("chr_rtlsim"):<13} | {str("chr_analytical"):<13} dif\n'

    rtlsim_size = 0
    chr_rtlsim_size = 0
    chr_analytical_size = 0
    
    for i in range(len(chr_rtlsim_list)):
        string += f'{str(chr_rtlsim_list[i][0] or "N/A"):<40} | {str(rtlsim_list[i][2]):<13} | {str(chr_rtlsim_list[i][2]):<13} | {str(chr_analytical_list[i][2]):<13} {chr_analytical_list[i][2]-chr_rtlsim_list[i][2]}\n'
        rtlsim_size += rtlsim_list[i][2]
        chr_rtlsim_size += chr_rtlsim_list[i][2]
        chr_analytical_size += chr_analytical_list[i][2]

    string += f'{str("Sum"):<40} | {str(rtlsim_size):<13} | {str(chr_rtlsim_size):<13} | {str(chr_analytical_size):<13} {chr_analytical_size-chr_rtlsim_size}\n'
    return string

In [4]:
from finn.builder.build_dataflow_steps import step_set_fifo_depths
import finn.builder.build_dataflow_config as build_cfg

In [ ]:
characterization_strategy_key = "analytical"
method_key = "characterize"
tmp_output_dir = root_dir

cfg = build_cfg.DataflowBuildConfig(
    output_dir=tmp_output_dir,
    auto_fifo_depths=True,
    auto_fifo_strategy=method_key,
    characteristic_function_strategy=characterization_strategy_key,
    target_fps=1000,
    force_python_rtlsim=False,
    synth_clk_period_ns=10.0,
    board="Pynq-Z1",
    rtlsim_batch_size=2,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
       # build_cfg.DataflowOutputType.STITCHED_IP,
        build_cfg.DataflowOutputType.RTLSIM_PERFORMANCE,
    ],
)

chr_rtlsim_model_path = f'/{root_dir}/VGG10_build_output/run_2_characterize_rtlsim/intermediate_models/step_generate_estimate_reports.onnx'

chr_rtlsim_model = ModelWrapper(chr_rtlsim_model_path)

chr_analytical_model_path = f'/{root_dir}/VGG10_build_output/run_3_characterize_analytical/intermediate_models/step_generate_estimate_reports.onnx'
chr_analytical_model = ModelWrapper(chr_analytical_model_path)


chr_analytical_model = step_set_fifo_depths(chr_analytical_model,cfg)

In [ ]:
print("original")
get_fifo_table(rtlsim_model_original, chr_rtlsim_model_original, chr_analytical_model_original)

#print("re-run step_fifo_sizing")
#get_fifo_table(rtlsim_model, chr_rtlsim_model, chr_analytical_model)

In [ ]:
#print("re-run step_fifo_sizing")

from finn.util.visualization import showInNetron
import onnx
from qonnx.util.basic import qonnx_make_model
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp
import os
from finn.analysis.fpgadataflow.dataflow_performance import dataflow_performance
from finn.transformation.fpgadataflow.insert_dwc import InsertDWC
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from qonnx.transformation.general import (
    ApplyConfig,
    GiveReadableTensorNames,
    GiveUniqueNodeNames,
    RemoveStaticGraphInputs,
    RemoveUnusedTensors,
)
from finn.transformation.fpgadataflow.annotate_cycles import AnnotateCycles
import qonnx.custom_op.registry as registry
from finn.util.fpgadataflow import is_hls_node, is_rtl_node

from finn.transformation.fpgadataflow.prepare_ip import PrepareIP, _codegen_single_node
from finn.transformation.fpgadataflow.prepare_rtlsim import PrepareRTLSim

# def _codegen_single_node(node, model, fpgapart, clk):
from finn.transformation.fpgadataflow.replace_verilog_relpaths import (
    ReplaceVerilogRelPaths,
)


from finn.transformation.fpgadataflow.derive_characteristic import (
    DeriveCharacteristic,
    DeriveFIFOSizes,
)


def prepare_model_for_fifo_testing(name="VGG10",path=None):

    chr_analytical_model_path = model_path
    chr_analytical_model = ModelWrapper(chr_analytical_model_path)

    part = "xcku3p-ffva676-1-e"
    clk_ns = 10.0

    model = chr_analytical_model
    model = model.transform(InsertDWC())
    model = model.transform(SpecializeLayers(part))
    model = model.transform(GiveUniqueNodeNames())
    model = model.transform(AnnotateCycles())

    #strategy = "analytical"
    strategy = "rtlsim"

    for node in model.graph.node:
        inst = registry.getCustomOp(node)
        if (is_hls_node(node) or is_rtl_node(node)) and (
            inst.prepare_kwargs_for_characteristic_fx() is None or strategy != "analytical"
        ):
            if inst.get_nodeattr("code_gen_dir_ipgen") == "":
                _codegen_single_node(
                    node, model, part, clk_ns
                )

            op_type = node.op_type
            if is_hls_node(node):
                try:
                    # lookup op_type in registry of CustomOps

                    # ensure that code is generated
                    assert (
                        inst.get_nodeattr("code_gen_dir_ipgen") != ""
                    ), """Node
                    attribute "code_gen_dir_ipgen" is empty. Please run
                    transformation PrepareIP first."""
                    if not os.path.isdir(
                        inst.get_nodeattr("ipgen_path")
                    ) or not inst.get_nodeattr("code_gen_dir_ipgen") in inst.get_nodeattr(
                        "ipgen_path"
                    ):
                        # call the compilation function for this node
                        inst.ipgen_singlenode_code()
                    else:
                        warnings.warn("Using pre-existing IP for %s" % node.name)
                    # ensure that executable path is now set
                    assert (
                        inst.get_nodeattr("ipgen_path") != ""
                    ), """Transformation
                    HLSSynthIP was not successful. Node attribute "ipgen_path"
                    is empty."""
                except KeyError:
                    # exception if op_type is not supported
                    raise Exception(
                        "Custom op_type %s is currently not supported." % op_type
                    )

    model = model.transform(ReplaceVerilogRelPaths())
    for node in model.graph.node:
        inst = registry.getCustomOp(node)
        if (is_hls_node(node) or is_rtl_node(node)) and (
            inst.prepare_kwargs_for_characteristic_fx() is None or strategy != "analytical"
        ):
            try:
                # lookup op_type in registry of CustomOps
                # inst = registry.getCustomOp(node)
                inst.prepare_rtlsim()
                # ensure that executable path is now set
                assert (
                    inst.get_nodeattr("rtlsim_so") != ""
                ), "Failed to prepare RTLSim, no rtlsim_so attribute found."
            except KeyError:
                # exception if op_type is not supported
                raise Exception("Custom op_type %s is currently not supported." % op_type)




    period = int(model.analysis(dataflow_performance)["max_cycles"] + 12)
    model = model.transform(
        DeriveCharacteristic(
            model,
            period,
            strategy,
            part,
            clk_ns,
        )
    )

    #model = model.transform(DeriveFIFOSizes())

    onnx_model2 = qonnx_make_model(model.graph, producer_name="simple-model2")
    onnx.save(onnx_model2, f'/{root_dir}/{name_of_model}_model_to_derive_rtlsim_original.onnx')

    dump = get_fifo_table(rtlsim_model_original, chr_rtlsim_model_original, model)
    print(dump)


name_of_model = "VGG10"
path = f'/{root_dir}/{name_of_model}_build_output/run_3_characterize_analytical/intermediate_models/step_generate_estimate_reports.onnx'

prepare_model_for_fifo_testing(name_of_model,path)

/home/lstasytis/fifo_sizing/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for layer StreamingMaxPool_hls_0 can be lower than
             actual latency!
  warnings.warn(
/home/lstasytis/fifo_sizing/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for layer StreamingMaxPool_hls_1 can be lower than
             actual latency!
  warnings.warn(
/home/lstasytis/fifo_sizing/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for layer StreamingMaxPool_hls_2 can be lower than
             actual latency!
  warnings.warn(
/home/lstasytis/fifo_sizing/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for layer StreamingMaxPool_hls_3 can be lower than
             actual latency!
  warnings.warn(
/home/lstasytis/fifo_sizing/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for lay

mvau vec shape 3:  1024
SF3:  1
NF3:  1
mvau vec shape 3:  512
SF3:  1
NF3:  2
mvau vec shape 3:  256
SF3:  1
NF3:  4
mvau vec shape 3:  128
SF3:  1
NF3:  8
mvau vec shape 3:  64
SF3:  1
NF3:  16
mvau vec shape 3:  32
SF3:  1
NF3:  32
mvau vec shape 3:  16
SF3:  1
NF3:  32
mvau vec shape 3:  1
SF3:  8
NF3:  64
mvau vec shape 3:  1
SF3:  4
NF3:  128
mvau vec shape 3:  1
SF3:  16
NF3:  24


%Warning-WIDTH: /tmp/finn_dev_lstasytis/code_gen_ipgen_FMPadding_rtl_0_tj9i2019/fmpadding.sv:116:19: Operator ASSIGN expects 1 bits on the Assign RHS, but Assign RHS's VARREF 'INIT_XEND' generates 32 bits.
                                                                                                   : ... In instance FMPadding_rtl_0.impl.padding
  116 |  xcount_t  XEnd = INIT_XEND;
      |                   ^~~~~~~~~
                ... For warning description see https://verilator.org/warn/WIDTH?v=4.224
                ... Use "/* verilator lint_off WIDTH */" and lint_on around source to disable this message.
%Warning-WIDTH: /tmp/finn_dev_lstasytis/code_gen_ipgen_FMPadding_rtl_0_tj9i2019/fmpadding.sv:117:19: Operator ASSIGN expects 1 bits on the Assign RHS, but Assign RHS's VARREF 'INIT_XON' generates 32 bits.
                                                                                                   : ... In instance FMPadding_rtl_0.impl.padding
  117 |  xcount_t  XOn  = I

make: Entering directory '/tmp/finn_dev_lstasytis/pyverilator_FMPadding_rtl_0_o8mrzi4o'
ccache g++  -I.  -MMD -I/usr/local/share/verilator/include -I/usr/local/share/verilator/include/vltstd -DVM_COVERAGE=0 -DVM_SC=0 -DVM_TRACE=1 -DVM_TRACE_FST=0 -DVM_TRACE_VCD=1 -faligned-new -fcf-protection=none -Wno-bool-operation -Wno-sign-compare -Wno-uninitialized -Wno-unused-but-set-variable -Wno-unused-parameter -Wno-unused-variable -Wno-shadow     -fPIC --std=c++11  -std=gnu++17 -Os -c -o pyverilator_wrapper.o /tmp/finn_dev_lstasytis/pyverilator_FMPadding_rtl_0_o8mrzi4o/pyverilator_wrapper.cpp
ccache g++  -I.  -MMD -I/usr/local/share/verilator/include -I/usr/local/share/verilator/include/vltstd -DVM_COVERAGE=0 -DVM_SC=0 -DVM_TRACE=1 -DVM_TRACE_FST=0 -DVM_TRACE_VCD=1 -faligned-new -fcf-protection=none -Wno-bool-operation -Wno-sign-compare -Wno-uninitialized -Wno-unused-but-set-variable -Wno-unused-parameter -Wno-unused-variable -Wno-shadow     -fPIC --std=c++11  -std=gnu++17 -Os -c -o verilated

%Warning-WIDTH: /tmp/finn_dev_lstasytis/code_gen_ipgen_ConvolutionInputGenerator_rtl_0_cn2n8eai/swg_common.sv:69:78: Operator ASSIGN expects 11 bits on the Assign RHS, but Assign RHS's VARREF 'LOOP_H_ITERATIONS' generates 32 bits.
                                                                                                                   : ... In instance ConvolutionInputGenerator_rtl_0.impl.controller_inst
   69 |     logic signed [$clog2(LOOP_H_ITERATIONS   +2)+1-1:0]  Counter_loop_h    = LOOP_H_ITERATIONS;
      |                                                                              ^~~~~~~~~~~~~~~~~
                ... For warning description see https://verilator.org/warn/WIDTH?v=4.224
                ... Use "/* verilator lint_off WIDTH */" and lint_on around source to disable this message.
%Warning-WIDTH: /tmp/finn_dev_lstasytis/code_gen_ipgen_ConvolutionInputGenerator_rtl_0_cn2n8eai/swg_common.sv:70:78: Operator ASSIGN expects 1 bits on the Assign RHS, but Assign R

...

make: Entering directory '/tmp/finn_dev_lstasytis/pyverilator_ChannelwiseOp_hls_0__e_p6re5'
ccache g++  -I.  -MMD -I/usr/local/share/verilator/include -I/usr/local/share/verilator/include/vltstd -DVM_COVERAGE=0 -DVM_SC=0 -DVM_TRACE=1 -DVM_TRACE_FST=0 -DVM_TRACE_VCD=1 -faligned-new -fcf-protection=none -Wno-bool-operation -Wno-sign-compare -Wno-uninitialized -Wno-unused-but-set-variable -Wno-unused-parameter -Wno-unused-variable -Wno-shadow     -fPIC --std=c++11  -std=gnu++17 -Os -c -o pyverilator_wrapper.o /tmp/finn_dev_lstasytis/pyverilator_ChannelwiseOp_hls_0__e_p6re5/pyverilator_wrapper.cpp
ccache g++  -I.  -MMD -I/usr/local/share/verilator/include -I/usr/local/share/verilator/include/vltstd -DVM_COVERAGE=0 -DVM_SC=0 -DVM_TRACE=1 -DVM_TRACE_FST=0 -DVM_TRACE_VCD=1 -faligned-new -fcf-protection=none -Wno-bool-operation -Wno-sign-compare -Wno-uninitialized -Wno-unused-but-set-variable -Wno-unused-parameter -Wno-unused-variable -Wno-shadow     -fPIC --std=c++11  -std=gnu++17 -Os -c -o v

%Warning-STMTDLY: /tmp/finn_dev_lstasytis/rtlsim_LabelSelect_hls_0_aek94q91/LabelSelect_hls_0.v:817:2: Unsupported: Ignoring delay on this delayed statement.
                                                                                                     : ... In instance LabelSelect_hls_0.grp_LabelSelect_hls_0_Pipeline_VITIS_LOOP_488_3_fu_45.flow_control_loop_pipe_sequential_init_U
  817 | #0 ap_loop_init_int = 1'b1;
      |  ^
                  ... For warning description see https://verilator.org/warn/STMTDLY?v=4.224
                  ... Use "/* verilator lint_off STMTDLY */" and lint_on around source to disable this message.
%Warning-STMTDLY: /tmp/finn_dev_lstasytis/rtlsim_LabelSelect_hls_0_aek94q91/LabelSelect_hls_0.v:818:2: Unsupported: Ignoring delay on this delayed statement.
                                                                                                     : ... In instance LabelSelect_hls_0.grp_LabelSelect_hls_0_Pipeline_VITIS_LOOP_488_3_fu_45.flow_cont

make: Entering directory '/tmp/finn_dev_lstasytis/pyverilator_LabelSelect_hls_0_46rv0x7y'
ccache g++  -I.  -MMD -I/usr/local/share/verilator/include -I/usr/local/share/verilator/include/vltstd -DVM_COVERAGE=0 -DVM_SC=0 -DVM_TRACE=1 -DVM_TRACE_FST=0 -DVM_TRACE_VCD=1 -faligned-new -fcf-protection=none -Wno-bool-operation -Wno-sign-compare -Wno-uninitialized -Wno-unused-but-set-variable -Wno-unused-parameter -Wno-unused-variable -Wno-shadow     -fPIC --std=c++11  -std=gnu++17 -Os -c -o pyverilator_wrapper.o /tmp/finn_dev_lstasytis/pyverilator_LabelSelect_hls_0_46rv0x7y/pyverilator_wrapper.cpp
ccache g++  -I.  -MMD -I/usr/local/share/verilator/include -I/usr/local/share/verilator/include/vltstd -DVM_COVERAGE=0 -DVM_SC=0 -DVM_TRACE=1 -DVM_TRACE_FST=0 -DVM_TRACE_VCD=1 -faligned-new -fcf-protection=none -Wno-bool-operation -Wno-sign-compare -Wno-uninitialized -Wno-unused-but-set-variable -Wno-unused-parameter -Wno-unused-variable -Wno-shadow     -fPIC --std=c++11  -std=gnu++17 -Os -c -o veril

/home/lstasytis/fifo_sizing/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for layer StreamingMaxPool_hls_6 can be lower than
             actual latency!
  warnings.warn(
/home/lstasytis/fifo_sizing/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for layer StreamingMaxPool_hls_5 can be lower than
             actual latency!
  warnings.warn(
/home/lstasytis/fifo_sizing/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for layer StreamingMaxPool_hls_4 can be lower than
             actual latency!
  warnings.warn(
/home/lstasytis/fifo_sizing/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for layer StreamingMaxPool_hls_3 can be lower than
             actual latency!
  warnings.warn(
/home/lstasytis/fifo_sizing/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for lay

inFIFODepths:
node                                     | rtlsim_large  | chr_rtlsim    | chr_analytical
FMPadding_rtl_0                          | 2             | 32            | 2            
ConvolutionInputGenerator_rtl_0          | 2             | 2             | 2            
MVAU_rtl_0                               | 2             | 2             | 2            
Thresholding_rtl_0                       | 2             | 2             | 2            
StreamingMaxPool_hls_0                   | 2             | 2             | 2            
FMPadding_rtl_1                          | 2             | 2             | 2            
ConvolutionInputGenerator_rtl_1          | 2             | 2             | 2            
MVAU_rtl_1                               | 2             | 2             | 2            
Thresholding_rtl_1                       | 2             | 2             | 2            
StreamingDataWidthConverter_rtl_0        | 2             | 2             | 2            
Stream

In [17]:
#the rerun of only the characterization and not the initial building

# store?
#onnx_model1 = qonnx_make_model(model.graph, producer_name="simple-model1")
#onnx.save(onnx_model1, f'/{root_dir}/vg10_model_to_derive.onnx')




import onnx
from qonnx.util.basic import qonnx_make_model
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp
import os

model = ModelWrapper(f'/{root_dir}/vg10_model_to_derive.onnx')
import importlib
import finn.transformation.fpgadataflow.derive_characteristic as derive_characteristic
import finn.transformation.fpgadataflow

# Reload from top-level to bottom-level
importlib.reload(finn.custom_op)
importlib.reload(finn.custom_op.fpgadataflow.matrixvectoractivation)
importlib.reload(finn.transformation.fpgadataflow)
importlib.reload(finn.transformation.fpgadataflow.derive_characteristic)

# Re-import the class
from finn.transformation.fpgadataflow.derive_characteristic import DeriveCharacteristic, DeriveFIFOSizes


period = int(model.analysis(dataflow_performance)["max_cycles"] + 12)
model = model.transform(
    DeriveCharacteristic(
        model,
        period,
        "analytical",
        part,
        clk_ns,
    )
)
model = model.transform(DeriveFIFOSizes())
dump = get_fifo_table(rtlsim_model_original, chr_rtlsim_model_original, model)
print(dump)

mvau vec shape 3:  1
SF3:  16
NF3:  24
mvau vec shape 3:  1
mvau vec shape 3:  SF3: 1 
16SF3: 
 NF3: 4 
24NF3:  
128
mvau vec shape 3:  1
SF3:  4
NF3:  128
mvau vec shape 3:  1
SF3:  8


/home/lstasytis/fifo_sizing/finn/src/finn/custom_op/fpgadataflow/hwcustomop.py:545: UserWarning: Skipping node StreamingMaxPool_hls_6: already has FIFO characteristic
  warnings.warn("Skipping node %s: already has FIFO characteristic" % self.onnx_node.name)


NF3:  64 
mvau vec shape 3: 1
SF3:  8
NF3:  64
mvau vec shape 3:  16
SF3:  1
NF3:  32
mvau vec shape 3:  16
SF3:  1
NF3:  32


/home/lstasytis/fifo_sizing/finn/src/finn/custom_op/fpgadataflow/hwcustomop.py:545: UserWarning: Skipping node ConvolutionInputGenerator_rtl_6: already has FIFO characteristic
  warnings.warn("Skipping node %s: already has FIFO characteristic" % self.onnx_node.name)
/home/lstasytis/fifo_sizing/finn/src/finn/custom_op/fpgadataflow/hwcustomop.py:545: UserWarning: Skipping node StreamingMaxPool_hls_5: already has FIFO characteristic
  warnings.warn("Skipping node %s: already has FIFO characteristic" % self.onnx_node.name)


mvau vec shape 3:  

...


SF3:  1
NF3:  2


/home/lstasytis/fifo_sizing/finn/src/finn/custom_op/fpgadataflow/hwcustomop.py:545: UserWarning: Skipping node StreamingMaxPool_hls_0: already has FIFO characteristic
  warnings.warn("Skipping node %s: already has FIFO characteristic" % self.onnx_node.name)


mvau vec shape 3:  1024
SF3:  1
NF3:  1
mvau vec shape 3:  1024
SF3:  1
NF3:  

/home/lstasytis/fifo_sizing/finn/src/finn/custom_op/fpgadataflow/hwcustomop.py:545: UserWarning: Skipping node ConvolutionInputGenerator_rtl_0: already has FIFO characteristic
  warnings.warn("Skipping node %s: already has FIFO characteristic" % self.onnx_node.name)


1


In [7]:
print(dump)

inFIFODepths:
node                                     | rtlsim_large  | chr_rtlsim    | chr_analytical
FMPadding_rtl_0                          | 2             | 32            | 32           
ConvolutionInputGenerator_rtl_0          | 2             | 2             | 2            
MVAU_rtl_0                               | 2             | 2             | 2            
Thresholding_rtl_0                       | 2             | 2             | 2            
StreamingMaxPool_hls_0                   | 2             | 2             | 2            
FMPadding_rtl_1                          | 2             | 2             | 2            
ConvolutionInputGenerator_rtl_1          | 2             | 2             | 2            
MVAU_rtl_1                               | 2             | 2             | 2            
Thresholding_rtl_1                       | 2             | 2             | 2            
StreamingDataWidthConverter_rtl_0        | 2             | 2             | 2            
Stream

In [ ]:
inst = getCustomOp(chr_analytical_model_original.graph.node[7])
print(chr_analytical_model_original.graph.node[7].name)
inst.get_nodeattr("outFIFODepths")

In [ ]:
inst.get_nodeattr("MW"), inst.get_nodeattr("MH"), inst.get_nodeattr("SIMD"), inst.get_nodeattr("PE")

In [ ]:
inst.get_nodeattr("numInputVectors")